# CNN Brain Playground
Define the CNN policy in-cell, register it inline, train for two generations.


In [1]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "train_headless.py").exists() and (REPO_ROOT.parent / "train_headless.py").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import math
import jax
import jax.numpy as jnp

from brains.config import load_runtime_spec
from brains.jax_trainer import apply_runtime_spec
from brains.models import (
    NotebookModel,
    apply_notebook_model,
    register_inline_policy_factory,
    register_notebook_model,
)
from brains.models.notebook_framework import train_and_save_with_progress


In [2]:
class CNNPatchPolicy:
    """Patch-pooled CNN-style policy: reshape obs into patches, encode, mean-pool, readout."""

    def __init__(self, input_size: int, output_size: int):
        if input_size % 4 != 0:
            raise ValueError("CNNPatchPolicy expects input_size divisible by 4.")
        self.input_size = input_size
        self.output_size = output_size
        self.patch_count = input_size // 4
        self.hidden = 12

    def init_params(self, key):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        return {
            "w_patch": jax.random.normal(k1, (self.hidden, 4), dtype=jnp.float32) * jnp.float32(1.0 / math.sqrt(4)),
            "b_patch": jax.random.uniform(k2, (self.hidden,), dtype=jnp.float32, minval=-0.2, maxval=0.2),
            "w_out": jax.random.normal(k3, (self.output_size, self.hidden), dtype=jnp.float32) * jnp.float32(1.0 / math.sqrt(self.hidden)),
            "b_out": jax.random.uniform(k4, (self.output_size,), dtype=jnp.float32, minval=-0.2, maxval=0.2),
        }

    def zero_state(self):
        return jnp.zeros((1,), dtype=jnp.float32)

    def step(self, params, state, obs, key, noise_scale):
        patches = obs.reshape((self.patch_count, 4))
        local = jnp.tanh(jnp.einsum("pf,hf->ph", patches, params["w_patch"]) + params["b_patch"])
        pooled = jnp.mean(local, axis=0)
        logits = jnp.matmul(params["w_out"], pooled) + params["b_out"]
        key, noise_key = jax.random.split(key)
        noisy = logits + (jax.random.normal(noise_key, (self.output_size,), dtype=jnp.float32) * jnp.maximum(noise_scale, 0.0))
        return state, jnp.tanh(noisy), key


def build_cnn_policy(spec, model_definition):
    del spec
    return CNNPatchPolicy(model_definition.input_size, model_definition.output_size)


CNN_ENTRYPOINT = register_inline_policy_factory("notebook_cnn_patch_v1", build_cnn_policy)
CNN_ENTRYPOINT


'inline:notebook_cnn_patch_v1'

In [3]:
MODEL_TYPE = "notebook_cnn_patch_v1"
LOG_ID = datetime.now(timezone.utc).strftime("cnn_patch_%Y%m%dT%H%M%SZ")
TRAIN_GENERATIONS = 2
SEED = 11

base_spec = load_runtime_spec(REPO_ROOT / "configs/default.yaml")
model = NotebookModel(
    model_type=MODEL_TYPE,
    architecture="cnn_patch",
    description="Patch-pooled CNN-style policy defined inline in this notebook.",
    input_size=48,
    output_size=4,
    parameter_count=None,
    policy_entrypoint=CNN_ENTRYPOINT,
    control_mode="motor_targets",
)
register_notebook_model(model, spec=base_spec, persist=False)

spec = apply_notebook_model(base_spec, model)
spec = replace(spec, name="cnn-patch-default")
apply_runtime_spec(spec)

{"model": spec.model.type, "architecture": spec.model.architecture, "control_mode": spec.control.mode}


W0501 11:10:38.474459 2651538 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


{'model': 'notebook_cnn_patch_v1',
 'architecture': 'cnn_patch',
 'control_mode': 'motor_targets'}

In [4]:
result = train_and_save_with_progress(
    spec=spec,
    repo_root=REPO_ROOT,
    model_type=MODEL_TYPE,
    log_id=LOG_ID,
    seed=SEED,
    generations=TRAIN_GENERATIONS,
    print_step_progress=False,
)
result


generation 1/2 start


generation 1/2 done | mean_reward=-19.7259 | best_reward=-59.7511


generation 2/2 start


generation 2/2 done | mean_reward=-96.5941 | best_reward=-16.4973


{'model_id': 'notebook_cnn_patch_v1_cnn_patch_20260501T151038Z',
 'log_id': 'cnn_patch_20260501T151038Z',
 'latest': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/checkpoints/notebook_cnn_patch_v1_cnn_patch_20260501T151038Z/latest.npz',
 'generation': 2,
 'mean_reward': -96.59406280517578,
 'best_reward': -16.497293930475855,
 'elapsed_s': 131.38992129098915,
 'visible_artifacts': ['notebook_cnn_patch_v1_cnn_patch_20260501T151038Z',
  'notebook_simple_snn_v1_simple_snn_20260501T151036Z',
  'notebook_command_brain_v1_command_brain_20260501T151039Z',
  'notebook_shared_trunk_v1_shared_trunk_20260501T130232Z']}